In [ ]:
%load_ext autoreload
%autoreload 2
import torch, rdkit
import os
import sys, pathlib
from pathlib import Path
_THIS_FILE = Path(globals().get("__file__", Path.cwd())).resolve()
PROJECT_ROOT = _THIS_FILE.parent.parent
os.environ["MODELS_VARIANT"] = "TransCVAE"
sys.path.insert(0, str(PROJECT_ROOT))
from rdkit import Chem
from utils.utils import *
from utils.eval import *
from utils.dataloader import *

device   = "cuda"
vocab = dataset.vocab
index_to_token = {idx: token for token, idx in vocab.items()}

In [ ]:
def select_model(choice, latent):
    if choice == "Trans_MHA":
        from models.Trans_MHA import CVAE, PriorNet
        model    = CVAE(latent_dim=latent).cuda().eval()
        model.decoder.cuda().eval()
        prior = PriorNet(y_dim=3, latent_dim=latent).cuda().eval()

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_dmodel256.pth")
        state_dict = torch.load(save_path)
        model.load_state_dict(state_dict)

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_prior.pth")
        state_dict = torch.load(save_path)
        prior.load_state_dict(state_dict)

    # Trans
    elif choice == "Trans":
        from models.Trans import CVAE, PriorNet
        model    = CVAE(latent_dim=latent).cuda().eval()
        model.decoder.cuda().eval()
        prior = PriorNet(y_dim=3, latent_dim=latent).cuda().eval()

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_dmodel256_no_mha.pth")
        state_dict = torch.load(save_path)
        model.load_state_dict(state_dict)

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_prior_no_mha.pth")
        state_dict = torch.load(save_path)
        prior.load_state_dict(state_dict)

    # LSTM
    elif choice == "LSTM":
        from models.LSTM import CVAE, PriorNet
        model    = CVAE(latent_dim=latent).cuda().eval()
        model.decoder.cuda().eval()
        prior = PriorNet(y_dim=3, latent_dim=latent).cuda().eval()

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_LSTM.pth")
        state_dict = torch.load(save_path)
        model.load_state_dict(state_dict)

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_LSTM_prior.pth")
        state_dict = torch.load(save_path)
        prior.load_state_dict(state_dict)


    # LSTM + MHA
    elif choice == "LSTM_MHA":
        from models.LSTM_MHA import CVAE, PriorNet
        model    = CVAE(latent_dim=latent).cuda().eval()
        model.decoder.cuda().eval()
        prior = PriorNet(y_dim=3, latent_dim=latent).cuda().eval()

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_LSTM_MHA.pth")
        state_dict = torch.load(save_path)
        model.load_state_dict(state_dict)

        save_path = (PROJECT_ROOT / "models/weights" / "model_weights_LSTM_MHA_prior.pth")
        state_dict = torch.load(save_path)
        prior.load_state_dict(state_dict)
    

    return model, prior

In [ ]:
mode = "LSTM_MHA"
latent_dim = 128
model, prior = select_model(mode, latent_dim)

In [ ]:
mu_qs = []
lv_qs = []
mu_ps = []
lv_ps = []
props = []

properties_results=[]
properties_origin=[]
print(len(val_dataset))
if mode == "Trans" or mode == "Trans_MHA":
    with torch.no_grad():
        for batch in eval_dataloader:
            (sm_enc, sm_dec_in, sm_dec_out, prop) = [t.to(device) for t in batch]
            _, _, mu_q, lv_q, _ = model(sm_enc, sm_dec_in, prop)
            mu_p, lv_p = prior.forward(prop.squeeze())

            mu_qs.append(mu_q); lv_qs.append(lv_q)
            mu_ps.append(mu_p); lv_ps.append(lv_p)
            props.append(prop)

else:
    for batch in eval_dataloader:
        with torch.no_grad():
            (_, sm_dec_in, sm_dec_out, prop) = [t.to(device) for t in batch]
            _, _, mu_q, lv_q, _ = model(sm_dec_in, sm_dec_out, prop)
            mu_p, lv_p = prior.forward(prop.squeeze())

            mu_qs.append(mu_q); lv_qs.append(lv_q)
            mu_ps.append(mu_p); lv_ps.append(lv_p)
            props.append(prop)

mu_q = torch.cat(mu_qs, dim=0); lv_q = torch.cat(lv_qs, dim=0)
mu_p = torch.cat(mu_ps, dim=0); lv_p = torch.cat(lv_ps, dim=0)
prop = torch.cat(props, dim=0)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import LogNorm
import umap.umap_ as UMAP

tsne = TSNE(n_components=2, perplexity=150, random_state=42)
umap = UMAP.UMAP(random_state=42, n_neighbors=300)

z_q = model.reparameterize(mu_q, lv_q)
z_p = model.reparameterize(mu_p, lv_p)



def reduce_dim(X:torch.Tensor):
    if mode != "LSTM":
        X = X.mean(dim=1).detach().cpu().numpy()
    else: X = X.detach().cpu().numpy()
    X_scaled = StandardScaler().fit_transform(X)
    X_pca    = PCA(n_components=100, random_state=42).fit_transform(X_scaled)
    reduced = tsne.fit_transform(X_pca)
    return reduced

reduced_mu_q = reduce_dim(mu_q)
reduced_mu_p = reduce_dim(mu_p)

reduced_z_q = reduce_dim(z_q)
reduced_z_p = reduce_dim(z_p)

if mode == "LSTM_MHA" or mode == "Trans_MHA":
    prop = model.input_embedding_p(prop)

    z_q_ff = model.alpha*model.crossattn(z_q, prop, prop) + z_q
    z_q_ff = model.norm1(model.ff(z_q_ff) + z_q_ff)

    z_p_ff = model.alpha*model.crossattn(z_p, prop, prop) + z_p
    z_p_ff = model.norm1(model.ff(z_p_ff) + z_p_ff)

    reduced_z_q_ff = reduce_dim(z_q_ff)
    reduced_z_p_ff = reduce_dim(z_p_ff)


In [ ]:
df = pd.read_csv(PROJECT_ROOT/"data/simulation-trajectory-aggregate_aligned.csv")
labels = df.iloc[:, 6].values.squeeze()
vmin, vmax = labels.min(), labels.max()  # 대개 -1.0, 0.5


def draw(reduced, labels, path, vmin=None, vmax=None):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        reduced[:, 0], reduced[:, 1],
        c=labels, cmap='viridis', s=10, alpha=0.7,
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Li+ Conductivity")

    ax.set_xlim(-60, 60)
    ax.set_ylim(-60, 60)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')  # ← 저장 먼저
    #plt.close(fig)  # 리소스 정리

    # 필요하면 아래 줄을 저장 이후에 호출
    plt.show()

save_path = PROJECT_ROOT/f"Plots/{mode}"
draw(reduced_mu_q, labels=labels, path=save_path/"mu_q.png", vmax=vmax, vmin=vmin)
draw(reduced_mu_p, labels, save_path/"mu_p.png", vmax, vmin)
draw(reduced_z_q, labels, save_path/"z_q.png", vmax, vmin)
draw(reduced_z_p, labels, save_path/"z_p.png", vmax, vmin)
if mode == "LSTM_MHA" or mode == "Trans_MHA":
    draw(reduced_z_q_ff, labels, save_path/"z_q_ff.png", vmax, vmin)
    draw(reduced_z_p_ff, labels, save_path/"z_p_ff.png", vmax, vmin)